# InsectSet459: Stream-and-Delete Feature Extraction Pipeline (Google Colab)

This notebook demonstrates the **stream-and-delete** feature extraction architecture:
1. Streams audio recordings directly from Zenodo zip archives via HTTP range requests into memory (`io.BytesIO`).
2. Decodes, chunks, and extracts 146-dimensional handcrafted features.
3. Checkpoints progress per-recording to Drive so runtime disconnects resume without re-downloading.
4. Immediately discards audio buffers, writing only `.npy` feature arrays directly to Google Drive.

**Storage Benefit:**
- 40-Species Stage: ~4.5 MB feature cache on Drive (vs 1.03 GB raw audio).
- Full 141-Species Benchmark: ~46 MB feature cache on Drive (vs 45.77 GB raw audio).
- Zero audio files stored on Google Drive or Colab local disk.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/insect_dl/data/processed/species40')
DRIVE_REFERENCE_DIR = Path('/content/drive/MyDrive/insect_audio/reference')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Target directory on Google Drive: {DRIVE_OUTPUT_DIR}")

## 2. Install Dependencies

In [ ]:
!pip install -q soundfile librosa soxr remotezip pandas requests tqdm scikit-learn

## 3. Clone Repository (feat/ap-stream-features branch)

In [ ]:
%cd /content
if not os.path.exists('insect-audio-dl'):
    !git clone -b feat/ap-stream-features https://github.com/ITWhizHarsh/insect-audio-dl.git
%cd /content/insect-audio-dl

## 4. Run Streaming Feature Extraction with Checkpoint Resume

In [ ]:
# If disconnected mid-run, re-executing this cell automatically resumes from the last completed file
!python scripts/colab_stream_features.py \
    --manifest data/meta/species40_manifest.csv \
    --drive-dir /content/drive/MyDrive/insect_dl/data/processed/species40 \
    --workers 4

## 5. Verify Reference Files SHA-256 Hashes

Validates that reference files uploaded to Google Drive match the exact local benchmark hashes.

In [ ]:
import hashlib

EXPECTED_HASHES = {
    'Train_hand.npy': 'ca35d97bd18ac791a3c7bc952e32978308c2c6cc971071d9d96bfb0b5aa4e946',
    'Train_index.csv': 'c9efe5a29099404e51a91a8d29e6703a01a0f3a95370d96c106541f5bba754c3',
    'Validation_hand.npy': '83b98553e25631a8e86f017e56deb309d628f4a27a65627394d69bc2027a6da2',
    'Validation_index.csv': 'cfb893e93c4a18331fa619aa1acefb7a73aee7eb978d7fc1886b0ffabaaa4017',
    'Test_hand.npy': '2f7f6af75b263d46a7caa13586f685e8974595b5a005598676418de2f4f0dec1',
    'Test_index.csv': '68d0e23bd7af0c55fa0611c91b36944fd9d36187d09774e81e6e87226c40887e'
}

print("=== Drive Reference SHA-256 Check ===")
all_hashes_ok = True
for fname, exp_hash in EXPECTED_HASHES.items():
    p = DRIVE_REFERENCE_DIR / fname
    if not p.exists():
        print(f"{fname:<20} : MISSING")
        all_hashes_ok = False
        continue
    actual_hash = hashlib.sha256(p.read_bytes()).hexdigest()
    status = "OK" if actual_hash == exp_hash else "MISMATCH"
    if status != "OK":
        all_hashes_ok = False
    print(f"{fname:<20} : {status}")

print("\nReference Upload Status:", "ALL OK" if all_hashes_ok else "VERIFICATION FAILED")

## 6. Parity Verification (vs Reference 40-Species Benchmark)

Compares the streamed feature arrays and chunk index files against reference files in Google Drive.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import soxr

print(f"Env: Python {sys.version.split()[0]} | numpy {np.__version__} | librosa {librosa.__version__} | soxr {soxr.__version__} | soundfile {sf.__version__} | torch {torch.__version__}")

splits = ['Train', 'Validation', 'Test']
all_array_equal = True
all_index_equal = True
overall_max_diff = 0.0

print("\n=== Parity Check vs Reference ===")
for split in splits:
    colab_hand = np.load(DRIVE_OUTPUT_DIR / f"{split}_hand.npy")
    ref_hand = np.load(DRIVE_REFERENCE_DIR / f"{split}_hand.npy")
    colab_idx = pd.read_csv(DRIVE_OUTPUT_DIR / f"{split}_index.csv")
    ref_idx = pd.read_csv(DRIVE_REFERENCE_DIR / f"{split}_index.csv")
    
    arr_eq = np.array_equal(colab_hand, ref_hand)
    idx_eq = colab_idx.equals(ref_idx)
    max_diff = float(np.max(np.abs(colab_hand - ref_hand)))
    
    if not arr_eq:
        all_array_equal = False
    if not idx_eq:
        all_index_equal = False
    if max_diff > overall_max_diff:
        overall_max_diff = max_diff
        
    print(f"{split:<10} | shape: {str(colab_hand.shape):<14} | array_equal: {str(arr_eq):<5} | max_abs_diff: {max_diff:.2e} | index_equal: {str(idx_eq):<5}")

if all_array_equal and all_index_equal:
    verdict = "PASS (all equal)"
elif all_index_equal and overall_max_diff < 1e-5:
    verdict = f"NEAR (not equal but max_abs_diff {overall_max_diff:.2e} below 1e-5)"
else:
    verdict = f"FAIL (array_equal={all_array_equal}, index_equal={all_index_equal}, max_abs_diff={overall_max_diff:.2e})"

print(f"\nVerdict: {verdict}")

## 7. Train MLP Baseline on Google Drive Data

In [ ]:
!python -m src.train \
    --model mlp \
    --data /content/drive/MyDrive/insect_dl/data/processed/species40 \
    --epochs 150 \
    --batch 64 \
    --wd 1e-4 \
    --seed 42 \
    --eval-test \
    --output /content/drive/MyDrive/insect_dl/results/mlp_species40_s42.json